# Databricks Pipeline Template — IT Cost Consolidation

Exemplo de pipeline Bronze → Silver → Gold com geração de dados sintéticos e exportação de relatórios gerenciais.

In [ ]:
import os
import logging
from typing import Optional
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("it_cost_pipeline")

In [ ]:
def ingest_bronze(spark, source_table, target_path, date_partition):
    """Lê dados de origem e escreve no Bronze (Delta)."""
    df = spark.read.format("jdbc").option("dbtable", source_table).load()
    df.write.format("delta").mode("append").partitionBy("load_date").option("path", target_path).save()
    print(f"Bronze: {source_table} → {target_path}")

In [ ]:
def generate_synthetic_bronze(spark, months: int = 12):
    """Gera um DataFrame bronze sintético para demonstração."""
    from pyspark.sql import Row
    import pandas as pd
    months_range = pd.date_range(end=pd.Timestamp.today(), periods=months, freq='MS')
    centers = [f"CC-{i:02d}" for i in range(1, 6)]
    categories = ["Software", "Hardware", "Cloud", "Consulting"]
    rows = []
    for m in months_range:
        for c in centers:
            for cat in categories:
                rows.append(Row(load_date=m.strftime("%Y-%m-%d"), cost_center=c, category=cat, vendor=f"VENDOR-{(hash(c+cat) % 10) + 1}", amount=round(10000 * (1 + 0.1 * centers.index(c)) * (1 + 0.05 * categories.index(cat)), 2), month=m.strftime("%Y-%m-01")))
    df = spark.createDataFrame(rows)
    logger.info("Synthetic bronze data generated (%d rows)", df.count())
    return df

In [ ]:
def load_silver_cost(spark, bronze_path, silver_path):
    """Cleansing e conformação do Bronze para Silver."""
    df = spark.read.format("delta").load(bronze_path)
    from pyspark.sql.functions import col, when, trim, upper, regexp_replace, abs as spark_abs
    silver = (df.filter(col("amount").isNotNull()).filter(col("amount") > 0).filter(col("amount") < 1_000_000_000).withColumn("cost_center", trim(upper(col("cost_center")))).withColumn("category", trim(upper(col("category")))).withColumn("amount", spark_abs(col("amount"))).dropDuplicates(["cost_center", "month", "category", "vendor"]))
    silver.write.format("delta").mode("overwrite").option("path", silver_path).save()
    print(f"Silver: {silver.count()} records written to {silver_path}")
    return silver

In [ ]:
def build_gold_aggregates(spark, silver_path, gold_path):
    """Gera agregados mensais usados pelo FP&A."""
    silver = spark.read.format("delta").load(silver_path)
    from pyspark.sql.functions import col, sum as spark_sum, avg as spark_avg
    monthly_by_cc = (silver.groupBy("month", "cost_center", "category").agg(spark_sum("amount").alias("total_amount"), spark_avg("amount").alias("avg_amount")).orderBy("month", "cost_center"))
    monthly_total = (silver.groupBy("month").agg(spark_sum("amount").alias("total_it_cost"), spark_avg("amount").alias("avg_cost_per_center")).orderBy("month"))
    monthly_by_cc.write.format("delta").mode("overwrite").option("path", f"{gold_path}/monthly_by_cc").save()
    monthly_total.write.format("delta").mode("overwrite").option("path", f"{gold_path}/monthly_total").save()
    print("Gold layer written:")
    print(f"  - {gold_path}/monthly_by_cc")
    print(f"  - {gold_path}/monthly_total")

In [ ]:
def generate_management_reports(spark, silver_df, out_dir: Optional[str] = None):
    """Exporta relatórios gerenciais (CSV) a partir do silver DataFrame."""
    import pandas as pd
    if out_dir is None:
        out_dir = os.path.join(os.path.dirname(__file__), '..', '..', 'reports')
    os.makedirs(out_dir, exist_ok=True)
    monthly_by_cc = (silver_df.groupBy("month", "cost_center").sum("amount").withColumnRenamed("sum(amount)", "total_amount").orderBy("month", "cost_center"))
    monthly_total = (silver_df.groupBy("month").sum("amount").withColumnRenamed("sum(amount)", "total_it_cost").orderBy("month"))
    try:
        mbc_pd = monthly_by_cc.toPandas()
        mt_pd = monthly_total.toPandas()
        mbc_path = os.path.join(out_dir, "monthly_by_cc.csv")
        mt_path = os.path.join(out_dir, "monthly_total.csv")
        mbc_pd.to_csv(mbc_path, index=False)
        mt_pd.to_csv(mt_path, index=False)
        logger.info("Management reports written: %s, %s", mbc_path, mt_path)
        return mbc_path, mt_path
    except Exception as e:
        logger.error("Failed to write management reports: %s", e)
        raise

## Uso

Exemplo de execução em um notebook Databricks: criar `ITCostPipeline(spark)` e chamar `run_full()`.